# Cars Compound — Train DAMAGE model (CarDD)

PC pe convert **ho chuka** hai. Ab Drive pe `cars-compound-yolo/damage` upload karo.

1. Runtime → Change runtime type → **GPU T4**
2. Cells Run all
3. Download `runs/damage/weights/best.onnx` → PC: `apps/ml-service/models/damage.onnx`

In [ ]:
!pip -q install ultralytics onnx onnxruntime
import torch
print('GPU:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU T4'

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/cars-compound-yolo/damage'
assert os.path.exists(DATA + '/data.yaml'), 'Upload damage folder to Drive first'
yaml = f"""path: {DATA}
train: images/train
val: images/val
names:
  0: dent
  1: scratch
  2: crack
  3: glass_shatter
  4: lamp_broken
  5: tire_flat
"""
open(DATA + '/data.yaml', 'w').write(yaml)
print('OK:', DATA)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n-seg.pt')
results = model.train(
    data=f'{DATA}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    project='runs',
    name='damage',
    exist_ok=True,
)

In [ ]:
from ultralytics import YOLO
best = YOLO('runs/damage/weights/best.pt')
metrics = best.val()
print('=== Accuracy (higher closer to 1.0 is better) ===')
try:
    print('mAP50-95:', float(metrics.box.map))
    print('mAP50:', float(metrics.box.map50))
except Exception as e:
    print('See runs/damage/ for charts', e)
best.export(format='onnx', simplify=True)
print('DOWNLOAD: runs/damage/weights/best.onnx')
print('Save on PC as apps/ml-service/models/damage.onnx')